In [ ]:
import plotly.graph_objects as go
from collections import defaultdict
import torch 
import torchvision 
import torch.nn as nn 
import torch.nn.functional as F  
from torch.utils.data import Subset
import torchvision.transforms.v2 as v2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms.functional as TF

import numpy as np 
import pandas as pd 
from tqdm import tqdm 
import matplotlib as mpl 
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split


base = torch.load("../models/full-ds.pt", weights_only=False)
base = base.to('mps')

augmented = torchvision.models.efficientnet_b0()
augmented.set_submodule('classifier.1', nn.Linear(in_features=1280, out_features=2))
augmented.load_state_dict(torch.load('../models/filtered-ds.pt', weights_only=True))
augmented = augmented.to('mps')

N_SAMPLES = 5
FORWARD_SIZE = 256

In [2]:
type(augmented)

torchvision.models.efficientnet.EfficientNet

In [3]:
transform = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32)
])

dataset = ImageFolder('../data', transform=transform)
idxs = list(range(len(dataset)))

train_idxs, test_idxs = train_test_split(idxs, stratify=dataset.targets)
train = Subset(dataset, train_idxs)
test = Subset(dataset, test_idxs)

In [4]:
base_activations = defaultdict(list)
aug_activations = defaultdict(list)

def get_hook(name, activations):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

for (name_b, module_b), (name_a, module_a) in zip(base.named_modules(), augmented.named_modules()):
    module_b.register_forward_hook(get_hook(name_b, base_activations))
    module_a.register_forward_hook(get_hook(name_a, aug_activations))

In [5]:
def linear_CKA(X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    def center_gram(K: torch.Tensor) -> torch.Tensor:
        n = K.size(0)
        H = torch.eye(n, device=K.device) - torch.ones(n, n, device=K.device) / n
        return H @ K @ H

    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    K = X @ X.T
    L = Y @ Y.T

    Kc = center_gram(K)
    Lc = center_gram(L)

    hsic = (Kc * Lc).sum()
    norm_x = (Kc * Kc).sum().sqrt()
    norm_y = (Lc * Lc).sum().sqrt()

    return hsic / (norm_x * norm_y + 1e-12)

In [6]:
sims = []

for a, sample in enumerate(test):
    torch.mps.empty_cache()
    if a > N_SAMPLES: 
        break

    sample, label = sample
    sample = sample.unsqueeze(0).to('mps')
    
    
    base(sample)
    augmented(sample)

         
    sim_matrix = torch.zeros((len(aug_activations), len(aug_activations)))
    loop = tqdm(total=len(aug_activations) ** 2, desc=f"{a + 1}/{N_SAMPLES} Computing RSA w/ CKA", leave=True)

    for i, (k_b,v_b) in enumerate(base_activations.items()):
        base_act = v_b[-1].reshape(v_b[-1].shape[0], -1)

        for j, (k_p,v_p) in enumerate(aug_activations.items()): 
            aug_act = v_p[-1].reshape(v_p[-1].shape[0], -1)
            sim_matrix[i, j] = linear_CKA(base_act.detach().flatten(), aug_act.detach().flatten())
            loop.update(1)
        
    sims.append(sim_matrix)

sim_matrix = torch.stack(sims).mean(dim=0)

1/5 Computing RSA w/ CKA:   0%|          | 0/108900 [00:00<?, ?it/s]/var/folders/jd/lb_vbvw54zd00wp54hlkx4_m0000gn/T/ipykernel_10546/4161545170.py:10: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4416.)
  K = X @ X.T


IndexError: Dimension specified as 0 but tensor has no dimensions

In [ ]:
base_activations

In [ ]:
def custom_hover_text(x, y, value):
    return f"CKA: {value:.3}<br>Base: {list(base_activations.keys())[x]}<br>Pretrained: {list(filtered_activations.keys())[y]}<br>({x}, {y})"

nrows, ncols = sim_matrix.shape
customdata = np.empty((nrows, ncols), dtype=object)

for i in range(nrows):
    for j in range(ncols):
        customdata[i, j] = custom_hover_text(i, j, sim_matrix[i, j])

fig = go.Figure(
    data=go.Heatmap(
        z=sim_matrix,
        colorscale='Viridis',
        customdata=customdata,
        hovertemplate='%{customdata}<extra></extra>',
        showscale=True,
    ), 
)

fig.update_layout(
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False, scaleanchor='x', scaleratio=1),
    plot_bgcolor='rgba(255,255,255,255)',
    paper_bgcolor='rgba(255,255,255,255)',
    margin=dict(t=30, b=10, l=10, r=10),
    title=dict(
        text=best_model.ID, 
        xref='paper', 
        yref='container',
        yanchor='top',
        x=0.5, 
        automargin=True,
        pad=dict(t=5)
    ),
)

fig.show()